# RAG with OpenAI Native

Lab for implementing Retrieval-Augmented Generation using OpenAI's native capabilities.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

print("OpenAI key loaded:", OPENAI_API_KEY is not None)
print("Pinecone key loaded:", PINECONE_API_KEY is not None)

OpenAI key loaded: True
Pinecone key loaded: True


In [2]:
from openai import OpenAI
from PyPDF2 import PdfReader
import tiktoken
import numpy as np

client = OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI client ready")

OpenAI client ready


In [3]:
pdf_path = "data/Dissertation.pdf"

reader = PdfReader(pdf_path)

pages = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text()
    if text:
        pages.append({
            "page": page_number,
            "text": text
        })

print(f"Pages with extracted text: {len(pages)}")
print(pages[0]["text"][:500])

Pages with extracted text: 44
30021853  DISSERTATION  MAY 2009  
 
Page | 0  
 
“THE ILLUMINATI AND THE TOTALITARIAN TIP -TOE” 
 
SHOULD CONSPIRACY THEORIES BE COVERED  IN THE MAINSTREAM MEDIA?  
 
 
 
 (Student Number – 30021853 ) 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
May 2009  
 
Word  count: 9,070  
 
 
 
 
Submitted in partial fulfillment of  
the requirements for the degree of B.A.(Hons)  
 
 
Media Studies Degree awarded by Liverpool Hope University and studied at 
Wirral Metropolitan College  
 
 
 
 
 
 
 
 
 
 
 
 
 
  


In [4]:
total_characters = sum(len(page["text"]) for page in pages)

print(f"Pages extracted: {len(pages)}")
print(f"Total characters: {total_characters:,}")

Pages extracted: 44
Total characters: 76,305


In [5]:
chunk_size = 1000
overlap = 200

chunks = []

for page in pages:
    text = page["text"]

    for i in range(0, len(text), chunk_size - overlap):
        chunk_text = text[i:i + chunk_size]

        if chunk_text.strip():
            chunks.append({
                "page": page["page"],
                "text": chunk_text
            })

print(f"Total chunks: {len(chunks)}")
print(chunks[0]["text"][:300])

Total chunks: 115
30021853  DISSERTATION  MAY 2009  
 
Page | 0  
 
“THE ILLUMINATI AND THE TOTALITARIAN TIP -TOE” 
 
SHOULD CONSPIRACY THEORIES BE COVERED  IN THE MAINSTREAM MEDIA?  
 
 
 
 (Student Number – 30021853 ) 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
May 2009  
 
Word  count: 9,070  
 
 
 
 
Submitted in partial fulf


In [7]:
print(chunks[10]["text"][:1000])

d to silence dissident 
government voices such as anti -war protesto rs.  
 
A documentary credited to the director Dylan Avery titled Loose Change became 
widely available over the internet and spread rapidly. The film gave an alternative 
version of the events of 9/11 and posed many interesting questions implying that the 
official version of the events of that fateful day was a lie and that it was an inside job.  
Loose Change  is one of many „investigative‟ documentaries on the subject of 9/11, 
the related concept of the New World Order, Illuminati and the insinuation of a long 
lost hidden knowledge that the majority of the world are kept from through the 
actions of this elite.  
 
A film claimed to riddled with so many inaccuracies and misinformation that it 
prompted one author to label it  
 
“perhaps the most factually challenged d ocumentary I‟ve ever seen...This 
is an amateurish piece of panic -peddling nonsense riddled with half -
truths, rehashed rumors, and implausible

In [8]:
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=chunks[10]["text"]
)

embedding = response.data[0].embedding

print(type(embedding))
print(len(embedding))
print(embedding[:10])

<class 'list'>
1536
[0.01192474365234375, -0.0011453628540039062, -0.029388427734375, 0.0287017822265625, -0.0460205078125, -0.0190582275390625, -0.0211334228515625, 0.0241851806640625, -0.042510986328125, -0.029937744140625]


In [9]:
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding


for chunk in chunks:
    chunk["embedding"] = get_embedding(chunk["text"])

print(f"Embeddings added to {len(chunks)} chunks")
print(len(chunks[0]["embedding"]))

Embeddings added to 115 chunks
1536


In [10]:
print(chunks[10].keys())

dict_keys(['page', 'text', 'embedding'])


In [11]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

print("Pinecone connected")

Pinecone connected


In [12]:
print(pc.list_indexes())

IndexList([])


In [13]:
from pinecone import ServerlessSpec

index_name = "rag-openai-native"

if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("Index creation requested")

Index creation requested


In [14]:
print(pc.list_indexes())

IndexList([<name='rag-openai-native', dim=1536, ready=True>])


In [15]:
index = pc.Index(index_name)

print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=0, metric='cosine', namespaces=0)


In [16]:
vectors = []

for i, chunk in enumerate(chunks):
    vectors.append({
        "id": f"chunk-{i}",
        "values": chunk["embedding"],
        "metadata": {
            "page": chunk["page"],
            "text": chunk["text"]
        }
    })

print(f"Prepared {len(vectors)} vectors")
print(vectors[0]["id"])

Prepared 115 vectors
chunk-0


In [17]:
index.upsert(vectors=vectors)

print("Vectors uploaded")

Vectors uploaded


In [18]:
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=115, metric='cosine', namespaces=1)


In [19]:
question = "What role did Loose Change play in conspiracy culture?"

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=question
)

question_embedding = response.data[0].embedding

print(len(question_embedding))
print(question_embedding[:10])

1536
[0.01457977294921875, -0.01403045654296875, -0.06414794921875, 0.03631591796875, -0.01959228515625, -0.031524658203125, -0.0199432373046875, 0.0267333984375, -0.025177001953125, 0.00502777099609375]


In [20]:
results = index.query(
    vector=question_embedding,
    top_k=3,
    include_metadata=True
)

print(results)

QueryResponse(matches=[ScoredVector(id='chunk-10', score=0.631386697, values=[], metadata={'page': 5, 'text': 'd to silence dissident \ngovernment voices such as anti -war protesto rs.  \n \nA documentary credited to the director Dylan Avery titled Loose Change became \nwidely available over the internet and spread rapidly. The film gave an alternative \nversion of the events of 9/11 and posed many interesting questions implying that the \nofficial version of the events of that fateful day was a lie and that it was an inside job.  \nLoose Change  is one of many „investigative‟ documentaries on the subject of 9/11, \nthe related concept of the New World Order, Illuminati and the insinuation of a long \nlost hidden knowledge that the majority of the world are kept from through the \nactions of this elite.  \n \nA film claimed to riddled with so many inaccuracies and misinformation that it \nprompted one author to label it  \n \n“perhaps the most factually challenged d ocumentary I‟ve eve

In [21]:
for match in results["matches"]:
    print(f"Chunk ID: {match['id']}")
    print(f"Score: {match['score']:.3f}")
    print(f"Page: {match['metadata']['page']}")
    print(match["metadata"]["text"][:500])
    print("-" * 80)

Chunk ID: chunk-10
Score: 0.631
Page: 5
d to silence dissident 
government voices such as anti -war protesto rs.  
 
A documentary credited to the director Dylan Avery titled Loose Change became 
widely available over the internet and spread rapidly. The film gave an alternative 
version of the events of 9/11 and posed many interesting questions implying that the 
official version of the events of that fateful day was a lie and that it was an inside job.  
Loose Change  is one of many „investigative‟ documentaries on the subject of 9/
--------------------------------------------------------------------------------
Chunk ID: chunk-95
Score: 0.451
Page: 36
ound within this examination of conspiracy theory were 
claims of false flag alien invasions, systematic poisoning of food supplies, 
holographic simulations of returns of religious prophets in order to bring about a one 
world religion, death camps and coffins being manufactured throughout the united 
states and a whole hos t of other

In [22]:
def retrieve_context(question, top_k=3):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    )
    
    question_embedding = response.data[0].embedding
    
    results = index.query(
        vector=question_embedding,
        top_k=top_k,
        include_metadata=True
    )
    
    return results["matches"]

In [23]:
test_question = "What role did Loose Change play in conspiracy culture?"

matches = retrieve_context(test_question, top_k=3)

for match in matches:
    print(f"Chunk ID: {match['id']}")
    print(f"Score: {match['score']:.3f}")
    print(f"Page: {match['metadata']['page']}")
    print(match["metadata"]["text"][:300])
    print("-" * 80)

Chunk ID: chunk-10
Score: 0.631
Page: 5
d to silence dissident 
government voices such as anti -war protesto rs.  
 
A documentary credited to the director Dylan Avery titled Loose Change became 
widely available over the internet and spread rapidly. The film gave an alternative 
version of the events of 9/11 and posed many interesting qu
--------------------------------------------------------------------------------
Chunk ID: chunk-95
Score: 0.451
Page: 36
ound within this examination of conspiracy theory were 
claims of false flag alien invasions, systematic poisoning of food supplies, 
holographic simulations of returns of religious prophets in order to bring about a one 
world religion, death camps and coffins being manufactured throughout the unit
--------------------------------------------------------------------------------
Chunk ID: chunk-6
Score: 0.430
Page: 4
 Roswell, New Mexico in 1947 to the assassination of 
Presi dent John F Kennedy in Dallas,  1963  through  to doubt 

In [24]:
def answer_question(question, top_k=3):
    matches = retrieve_context(question, top_k=top_k)
    
    context = "\n\n".join(
        [
            f"Source page {match['metadata']['page']}:\n{match['metadata']['text']}"
            for match in matches
        ]
    )
    
    prompt = f"""
Use the following context to answer the question.

If the answer is not in the context, say you do not know.

Context:
{context}

Question:
{question}
"""
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You answer questions using only the provided context and cite page numbers."},
            {"role": "user", "content": prompt}
        ]
    )
    
    return response.choices[0].message.content

In [25]:
answer = answer_question("What role did Loose Change play in conspiracy culture?")
print(answer)

Loose Change played a significant role in conspiracy culture by presenting an alternative version of the events of 9/11, suggesting that the official narrative was a lie and that the attacks were an inside job. The documentary became widely available on the internet and spread rapidly, contributing to the discourse surrounding conspiracy theories related to 9/11, the New World Order, and elite control of hidden knowledge (source page 5). It is noted for being "riddled with inaccuracies and misinformation," which further fueled debates and enhanced the visibility of conspiracy theories during that time (source page 5).


In [26]:
print(answer_question("Was David Icke correct in any of his theories?"))

The provided context does not indicate whether David Icke was correct in any of his theories. It mentions his beliefs and the ridicule he faced, but it does not provide evidence or conclusions about the correctness of his theories. Therefore, I do not know.


In [27]:
print(answer_question("What role did David Icke play in conspiracy culture?"))

David Icke is portrayed as a significant figure in conspiracy culture, particularly noted for his controversial claims made on the BBC chat show Wogan, where he declared himself the "son of God" and asserted that collective thoughts influence reality. This moment marked him as a target of ridicule, which he claims contributed to his personal development and prominence in the conspiracy discourse. Over time, attitudes toward him shifted, culminating in a TV documentary titled "David Icke: Was he right?" which reflects a broader change in public consciousness regarding alternative viewpoints, including conspiracy theories (source page 4).


In [ ]:
index = pc.Index(index_name)

print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=0, metric='cosine', namespaces=0)


## Example Questions and Answers

In [28]:
questions = [
    "What role did Loose Change play in conspiracy culture?",
    "What role did David Icke play in conspiracy culture?",
    "Who won the Premier League in 2025?"
]

for question in questions:
    print("Question:", question)
    print(answer_question(question))
    print("-" * 80)

Question: What role did Loose Change play in conspiracy culture?
Loose Change played a significant role in conspiracy culture by providing an alternative version of the events of 9/11, suggesting that the official narrative was a lie and that it may have been an inside job. The film gained widespread availability on the internet, contributing to its rapid spread and influence. It is one of many "investigative" documentaries that explore conspiracy theories related to 9/11, the New World Order, and hidden knowledge kept from the public by an elite group. However, it has been criticized for containing numerous inaccuracies and misinformation, with one author describing it as "perhaps the most factually challenged documentary I’ve ever seen" (page 5).
--------------------------------------------------------------------------------
Question: What role did David Icke play in conspiracy culture?
David Icke played a significant role in conspiracy culture by presenting alternative viewpoints t